# Complete Guide to Distribution Shift Detection

Learn to detect when your deployment data differs from training data.

**What you'll learn:**
- What distribution shift is and why it matters
- 4 covariate shift detectors: MMD, Energy, KS Test, Classifier (BBSD)
- Label shift detection and correction
- Importance weighting for covariate shift adaptation
- 5 distance metrics for measuring shift
- 5 visualization methods: feature histograms, embedding space, confidence distributions, KS statistics, shift severity
- How model performance degrades under shift
- Production monitoring and alerting

**Runtime:** ~5 min (MPS/CUDA), ~10 min (CPU)

## Setup

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Shift detection methods
from incerto.shift import (
    MMDShiftDetector,
    EnergyShiftDetector,
    KSShiftDetector,
    ClassifierShiftDetector,
    LabelShiftDetector,
    ImportanceWeightingShift,
)

# Metrics
from incerto.shift import (
    energy_distance,
    wasserstein_distance,
    sliced_wasserstein_distance,
    total_variation,
    population_stability_index,
)

# Visualization
from incerto.shift import (
    plot_feature_histograms,
    plot_embedding_space,
    plot_confidence_distributions,
    plot_shift_severity,
    plot_ks_statistics,
)

from incerto.utils import ConvNet, seed_everything

seed_everything(42)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## Part 1: What is Distribution Shift?

**Distribution shift** occurs when test/production data differs from training data.

**Types:**
- **Covariate shift:** P(X) changes, P(Y|X) stays same (e.g., camera angle changes)
- **Label shift:** P(Y) changes, P(X|Y) stays same (e.g., class imbalance changes)
- **Concept drift:** P(Y|X) changes (e.g., relationship between features and labels changes)

**Why detect it?**
- Model performance degrades silently
- Need to retrain or adapt model
- Safety and reliability in production

**Example:** Model trained on summer images fails on winter images

## Part 2: Load Data - Normal vs Shifted

We use **Fashion-MNIST** with two types of shift:
1. **Covariate shift**: Rotated images (simulates camera angle change)
2. **Label shift**: Different class distribution (simulates seasonal trends)

In [ ]:
# Reference distribution: Normal Fashion-MNIST
transform_normal = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

# Shifted distribution: Rotated images (simulates camera angle change)
transform_rotated = transforms.Compose([
    transforms.RandomRotation(degrees=45),
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform_normal)
test_normal = datasets.FashionMNIST('./data', train=False, download=True, transform=transform_normal)
test_rotated = datasets.FashionMNIST('./data', train=False, download=True, transform=transform_rotated)

# Split: train (40k), validation (10k for label shift), calibration (10k)
train_subset, val_subset, cal_subset = random_split(train_dataset, [40000, 10000, 10000])
test_normal_subset = Subset(test_normal, range(2000))
test_rotated_subset = Subset(test_rotated, range(2000))

# Optimized data loaders
num_workers = min(4, os.cpu_count() or 0)
pin_memory = device.type == "cuda"
loader_kwargs = dict(num_workers=num_workers, pin_memory=pin_memory, persistent_workers=num_workers > 0)

train_loader = DataLoader(train_subset, batch_size=256, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_subset, batch_size=256, shuffle=False, **loader_kwargs)
cal_loader = DataLoader(cal_subset, batch_size=256, shuffle=False, **loader_kwargs)
test_normal_loader = DataLoader(test_normal_subset, batch_size=256, shuffle=False, **loader_kwargs)
test_rotated_loader = DataLoader(test_rotated_subset, batch_size=256, shuffle=False, **loader_kwargs)

print(f"Reference (train): {len(train_subset)}")
print(f"Validation: {len(val_subset)}")
print(f"Test (no shift): {len(test_normal_subset)}")
print(f"Test (covariate shift): {len(test_rotated_subset)}")
print(f"DataLoader: num_workers={num_workers}, pin_memory={pin_memory}")

In [ ]:
# Visualize what the shift looks like
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

# Get some samples (without normalization for display)
display_transform_normal = transforms.ToTensor()
display_transform_rotated = transforms.Compose([
    transforms.RandomRotation(degrees=45),
    transforms.ToTensor(),
])

display_normal = datasets.FashionMNIST('./data', train=False, transform=display_transform_normal)
display_rotated = datasets.FashionMNIST('./data', train=False, transform=display_transform_rotated)

CLASS_NAMES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

torch.manual_seed(42)
for i in range(5):
    idx = i * 100  # Sample different images
    img_normal, label = display_normal[idx]
    img_rotated, _ = display_rotated[idx]
    
    axes[0, i].imshow(img_normal.squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(CLASS_NAMES[label], fontsize=10)
    
    axes[1, i].imshow(img_rotated.squeeze(), cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel("Normal", fontsize=12)
axes[1, 0].set_ylabel("Rotated (45°)", fontsize=12)
fig.suptitle("Example Images: Normal vs Covariate Shift (Rotation)", fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("The rotated images simulate a camera angle change in production.")

## Part 3: Train Feature Extractor

In [ ]:
# Train model for feature extraction and classification
model = ConvNet(num_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

model.train()
print("Training model (5 epochs on Fashion-MNIST)...")
for epoch in range(5):
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    accuracy = 100. * correct / total
    print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Accuracy = {accuracy:.2f}%")

print("Done!")

## Part 4: Extract Model Features

For shift detection, we use **model embeddings** rather than raw pixels because:
1. Lower dimensionality (64 vs 784) - more efficient
2. Semantically meaningful features
3. Better suited for kernel methods like MMD

We extract features from the penultimate layer of the trained CNN.

In [ ]:
# Extract model features (embeddings from penultimate layer)
def extract_model_features(model, loader, device):
    """Extract features from the model's penultimate layer."""
    model.eval()
    features = []
    labels_list = []
    
    # Hook to capture features before the final layer
    activation = {}
    
    # Register hook on the final fc layer to capture its input (128-dim for ConvNet)
    # ConvNet architecture: conv layers -> flatten -> fc1(128) -> relu -> dropout -> fc2(num_classes)
    handle = model.fc2.register_forward_hook(lambda m, inp, out: activation.update({'features': inp[0].detach()}))
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            _ = model(inputs)  # Forward pass to trigger hook
            features.append(activation['features'].cpu())
            labels_list.append(labels)
    
    handle.remove()
    return torch.cat(features), torch.cat(labels_list)

# Extract features from all datasets
print("Extracting model features...")
cal_features, cal_labels = extract_model_features(model, cal_loader, device)
test_normal_features, test_normal_labels = extract_model_features(model, test_normal_loader, device)
test_rotated_features, test_rotated_labels = extract_model_features(model, test_rotated_loader, device)

print(f"Feature shape: {cal_features.shape}")  # Should be (N, 128) for ConvNet
print(f"  Calibration: {cal_features.shape[0]} samples")
print(f"  Test normal: {test_normal_features.shape[0]} samples")
print(f"  Test rotated: {test_rotated_features.shape[0]} samples")

In [ ]:
# Create feature loaders for shift detectors
from torch.utils.data import TensorDataset

cal_feature_loader = DataLoader(
    TensorDataset(cal_features, cal_labels),
    batch_size=256, shuffle=False
)
test_normal_feature_loader = DataLoader(
    TensorDataset(test_normal_features, test_normal_labels),
    batch_size=256, shuffle=False
)
test_rotated_feature_loader = DataLoader(
    TensorDataset(test_rotated_features, test_rotated_labels),
    batch_size=256, shuffle=False
)

print("Feature loaders created")

## Part 5: Covariate Shift Detection

Covariate shift detectors compare feature distributions between reference and test data.

| Detector | Method | Best for |
|----------|--------|----------|
| `MMDShiftDetector` | Maximum Mean Discrepancy (kernel) | Non-parametric, any dimension |
| `EnergyShiftDetector` | Energy distance | Similar to MMD, no kernel choice |
| `KSShiftDetector` | Kolmogorov-Smirnov test | Per-feature, interpretable |
| `ClassifierShiftDetector` | Train classifier to separate | High dimensions, flexible |

In [ ]:
# Fit all covariate shift detectors on model features
# Note: For MMD, sigma should be ~sqrt(median_pairwise_distance / 2)
# For 128-dim normalized features, sigma ~10 works well

detectors = {
    "MMD": MMDShiftDetector(sigma=10.0),  # Larger sigma for 128-dim features
    "Energy": EnergyShiftDetector(),
    "KS": KSShiftDetector(),
    "Classifier": ClassifierShiftDetector(),
}

print("Fitting detectors on model features...")
for name, detector in detectors.items():
    detector.fit(cal_feature_loader)
    print(f"  {name}: {repr(detector)}")

In [ ]:
# Score on no-shift and shifted data
print("\nCovariate Shift Detection Results:")
print("=" * 70)
print(f"{'Detector':<15} {'No Shift':<15} {'With Shift':<15} {'Ratio':<10} {'Alert'}")
print("-" * 70)

results = {}
for name, detector in detectors.items():
    score_normal = detector.score(test_normal_feature_loader)
    score_shifted = detector.score(test_rotated_feature_loader)
    ratio = score_shifted / (score_normal + 1e-10)
    alert = "WARNING" if ratio > 1.5 else "OK"
    
    results[name] = {
        "normal": score_normal,
        "shifted": score_shifted,
        "ratio": ratio,
    }
    
    print(f"{name:<15} {score_normal:<15.6f} {score_shifted:<15.6f} {ratio:<10.2f} {alert}")

print("=" * 70)

In [ ]:
# Visualize confidence distributions under shift
model.eval()
with torch.no_grad():
    # Get model predictions and confidences
    normal_confs = []
    shifted_confs = []
    
    for inputs, _ in test_normal_loader:
        outputs = model(inputs.to(device))
        probs = torch.softmax(outputs, dim=1)
        normal_confs.append(probs.max(dim=1).values.cpu())
    
    for inputs, _ in test_rotated_loader:
        outputs = model(inputs.to(device))
        probs = torch.softmax(outputs, dim=1)
        shifted_confs.append(probs.max(dim=1).values.cpu())
    
    normal_confs = torch.cat(normal_confs)
    shifted_confs = torch.cat(shifted_confs)

print(f"Confidence under no shift:   mean={normal_confs.mean():.3f}, std={normal_confs.std():.3f}")
print(f"Confidence under shift:      mean={shifted_confs.mean():.3f}, std={shifted_confs.std():.3f}")

# Use the library's visualization function
fig = plot_confidence_distributions(normal_confs, shifted_confs, show=False)
fig.axes[0].set_title("Model Confidence Under Distribution Shift", fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNote: Lower confidence under shift indicates the model is less certain about rotated images.")

In [ ]:
# Per-feature KS statistics: which embedding dimensions shifted most?
from scipy.stats import ks_2samp

# Compute KS statistic for each feature dimension
ks_stats = []
p_values = []
for i in range(cal_features.shape[1]):
    stat, pval = ks_2samp(cal_features[:, i].numpy(), test_rotated_features[:, i].numpy())
    ks_stats.append(stat)
    p_values.append(pval)

ks_stats = torch.tensor(ks_stats)

print("Per-feature KS statistics (128 embedding dimensions):")
print(f"  Mean KS:  {ks_stats.mean():.4f}")
print(f"  Max KS:   {ks_stats.max():.4f}")
print(f"  Features with KS > 0.1: {(ks_stats > 0.1).sum().item()}")
print(f"  Features with KS > 0.2: {(ks_stats > 0.2).sum().item()}")

# Use the library's visualization function
fig = plot_ks_statistics(ks_stats, top_k=15, show=False)
fig.axes[0].set_title("Most Shifted Embedding Dimensions (KS Statistic)", fontweight='bold')
plt.tight_layout()
plt.show()

print("\nHigher KS statistics indicate features that changed most under distribution shift.")

## Part 6: Distance Metrics

`incerto` provides 5 metrics for measuring distribution distance:

| Metric | Description | Use case |
|--------|-------------|----------|
| `energy_distance` | Szekely & Rizzo energy distance | General purpose |
| `wasserstein_distance` | Optimal transport (Sinkhorn) | Geometry-aware |
| `sliced_wasserstein_distance` | Fast 1D projections | High dimensions |
| `total_variation` | TVD for discrete distributions | Label distributions |
| `population_stability_index` | PSI (credit scoring) | Binned features |

In [ ]:
# Use model features for metric computation (already extracted above)
# Use smaller subsets for expensive metrics
ref_small = cal_features[:500]
test_normal_small = test_normal_features[:500]
test_shifted_small = test_rotated_features[:500]

print(f"Using model features: {ref_small.shape}")

In [ ]:
# Compute distance metrics
print("Distance Metrics:")
print("=" * 65)
print(f"{'Metric':<25} {'No Shift':<15} {'With Shift':<15} {'Ratio'}")
print("-" * 65)

# Use smaller subsets for expensive metrics (using model features extracted earlier)
ref_small = cal_features[:500]
test_normal_small = test_normal_features[:500]
test_shifted_small = test_rotated_features[:500]

metric_results = {}

# Energy distance
e_normal = energy_distance(ref_small, test_normal_small)
e_shifted = energy_distance(ref_small, test_shifted_small)
metric_results["Energy Distance"] = (e_normal, e_shifted)
print(f"{'Energy Distance':<25} {e_normal:<15.4f} {e_shifted:<15.4f} {e_shifted/e_normal:.2f}x")

# Sliced Wasserstein (faster than full Wasserstein)
sw_normal = sliced_wasserstein_distance(ref_small, test_normal_small, num_projections=50, seed=42)
sw_shifted = sliced_wasserstein_distance(ref_small, test_shifted_small, num_projections=50, seed=42)
metric_results["Sliced Wasserstein"] = (sw_normal, sw_shifted)
print(f"{'Sliced Wasserstein':<25} {sw_normal:<15.4f} {sw_shifted:<15.4f} {sw_shifted/sw_normal:.2f}x")

# Wasserstein (Sinkhorn, use even smaller subset)
w_normal = wasserstein_distance(ref_small[:200], test_normal_small[:200], max_iter=50)
w_shifted = wasserstein_distance(ref_small[:200], test_shifted_small[:200], max_iter=50)
metric_results["Wasserstein (Sinkhorn)"] = (w_normal, w_shifted)
print(f"{'Wasserstein (Sinkhorn)':<25} {w_normal:<15.4f} {w_shifted:<15.4f} {w_shifted/(w_normal+1e-6):.2f}x")

print("=" * 65)

In [ ]:
# Discrete metrics for label distributions
print("\nDiscrete Distribution Metrics (for label shift):")
print("=" * 50)

# Create example distributions
source_dist = torch.tensor([0.1] * 10)  # Uniform
target_uniform = torch.tensor([0.1] * 10)  # No shift
target_shifted = torch.tensor([0.3, 0.2, 0.1, 0.1, 0.1, 0.05, 0.05, 0.05, 0.025, 0.025])  # Shifted

tvd_no = total_variation(source_dist, target_uniform)
tvd_shift = total_variation(source_dist, target_shifted)
print("Total Variation Distance:")
print(f"  No shift:   {tvd_no:.4f}")
print(f"  With shift: {tvd_shift:.4f}")

psi_no = population_stability_index(source_dist, target_uniform)
psi_shift = population_stability_index(source_dist, target_shifted)
print("\nPopulation Stability Index (PSI):")
print(f"  No shift:   {psi_no:.4f}")
print(f"  With shift: {psi_shift:.4f}")
print("  (PSI > 0.25 typically indicates significant shift)")

## Part 7: Visualizing Distribution Shift

`incerto` provides five visualization functions:
- `plot_feature_histograms`: Compare feature distributions
- `plot_embedding_space`: t-SNE or PCA visualization
- `plot_confidence_distributions`: Compare model confidence under shift
- `plot_ks_statistics`: Per-feature KS statistics (which features shifted most)
- `plot_shift_severity`: Shift scores vs severity (for monitoring dashboards)

In [ ]:
# Feature histograms (using model embeddings)
fig = plot_feature_histograms(
    ref_small[:300],
    test_shifted_small[:300],
    feature_ids=[0, 10, 20, 30, 40],  # Sample 5 embedding dimensions
    bins=30,
    show=False
)
fig.suptitle("Feature Distribution Comparison (Reference vs Shifted)", y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Embedding space visualization (PCA is faster than t-SNE)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# No shift
fig1 = plot_embedding_space(
    ref_small[:200],
    test_normal_small[:200],
    method="pca",
    show=False
)
ax1 = fig1.axes[0]

# With shift
fig2 = plot_embedding_space(
    ref_small[:200],
    test_shifted_small[:200],
    method="pca",
    show=False
)
ax2 = fig2.axes[0]

# Copy to combined figure
for i, (src_ax, title) in enumerate([(ax1, "No Shift"), (ax2, "With Covariate Shift")]):
    for collection in src_ax.collections:
        axes[i].scatter(
            collection.get_offsets()[:, 0],
            collection.get_offsets()[:, 1],
            s=5, alpha=0.5,
            label=collection.get_label()
        )
    axes[i].set_title(title, fontweight='bold')
    axes[i].legend()
    axes[i].set_xlabel("PC1")
    axes[i].set_ylabel("PC2")

plt.close(fig1)
plt.close(fig2)
plt.suptitle("PCA Embedding Space Comparison", fontweight='bold')
plt.tight_layout()
plt.show()

## Part 8: Label Shift Detection

**Label shift** occurs when P(Y) changes but P(X|Y) stays the same. This is common in:
- Seasonal trends (e.g., more coats sold in winter)
- Demographic changes
- Market shifts

`LabelShiftDetector` estimates the target label distribution using confusion matrix correction.

In [ ]:
# Create label-shifted test set (more T-shirts and Trousers, fewer Bags)
def create_label_shifted_loader(dataset, class_weights, n_samples=2000):
    """Create a loader with shifted label distribution."""
    # Normalize weights
    weights = torch.tensor(class_weights, dtype=torch.float)
    weights = weights / weights.sum()
    
    # Group indices by class
    class_indices = {i: [] for i in range(10)}
    for idx in range(len(dataset)):
        _, label = dataset[idx]
        class_indices[label].append(idx)
    
    # Sample according to weights
    selected_indices = []
    samples_per_class = (weights * n_samples).int()
    
    for cls, n in enumerate(samples_per_class):
        n = min(int(n), len(class_indices[cls]))
        selected_indices.extend(np.random.choice(class_indices[cls], n, replace=False).tolist())
    
    return DataLoader(Subset(dataset, selected_indices), batch_size=256, shuffle=False, **loader_kwargs)

# Source: uniform distribution
source_weights = [1.0] * 10

# Target: shifted (more T-shirts/Trousers, fewer accessories)
target_weights = [3.0, 3.0, 1.0, 1.0, 1.0, 0.5, 1.0, 0.5, 0.5, 0.5]

label_shifted_loader = create_label_shifted_loader(test_normal, target_weights)
print(f"Label-shifted test set: {len(label_shifted_loader.dataset)} samples")

In [ ]:
# Fit label shift detector
label_detector = LabelShiftDetector(num_classes=10)
label_detector.fit(model, train_loader, val_loader)

print(f"Source label distribution: {label_detector.source_label_dist.numpy().round(3)}")

In [ ]:
# Estimate target distribution and measure shift
estimated_target_dist = label_detector.estimate_target_distribution(model, label_shifted_loader)

print("Label Shift Detection:")
print("=" * 60)
print(f"{'Class':<12} {'Source':<12} {'Est. Target':<12} {'True Target'}")
print("-" * 60)

CLASS_NAMES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

true_target = torch.tensor(target_weights) / sum(target_weights)

for i, name in enumerate(CLASS_NAMES):
    src = label_detector.source_label_dist[i].item()
    est = estimated_target_dist[i].item()
    true = true_target[i].item()
    print(f"{name:<12} {src:<12.3f} {est:<12.3f} {true:.3f}")

print("=" * 60)

# Compute shift magnitude
shift_tvd = label_detector.compute_shift_magnitude(model, label_shifted_loader, metric='tvd')
shift_kl = label_detector.compute_shift_magnitude(model, label_shifted_loader, metric='kl')

print(f"\nShift magnitude (TVD): {shift_tvd:.4f}")
print(f"Shift magnitude (KL):  {shift_kl:.4f}")

In [ ]:
# Visualize label distributions
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(10)
width = 0.25

ax.bar(x - width, label_detector.source_label_dist.numpy(), width, label='Source', alpha=0.8)
ax.bar(x, estimated_target_dist.numpy(), width, label='Estimated Target', alpha=0.8)
ax.bar(x + width, true_target.numpy(), width, label='True Target', alpha=0.8)

ax.set_xlabel('Class')
ax.set_ylabel('Proportion')
ax.set_title('Label Shift Detection: Source vs Target Distribution', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 9: Importance Weighting for Covariate Shift Adaptation

When covariate shift is detected, we can adapt by **reweighting** training samples:

$$w(x) = \frac{p_{target}(x)}{p_{source}(x)}$$

`ImportanceWeightingShift` estimates these weights using:
- `logistic`: Train classifier to separate source/target
- `kernel`: Kernel Mean Matching (KMM)

In [ ]:
# Estimate importance weights
iw = ImportanceWeightingShift(method='logistic', alpha=0.01)
iw.fit(ref_small[:500], test_shifted_small[:500])

weights = iw.compute_weights(ref_small[:500])

print(f"Importance Weighting: {repr(iw)}")
print("\nWeight statistics:")
print(f"  Mean:   {weights.mean():.4f} (should be ~1.0)")
print(f"  Std:    {weights.std():.4f}")
print(f"  Min:    {weights.min():.4f}")
print(f"  Max:    {weights.max():.4f}")

In [ ]:
# Visualize importance weights
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of weights
axes[0].hist(weights.numpy(), bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(x=1.0, color='red', linestyle='--', label='No shift (w=1)')
axes[0].set_xlabel('Importance Weight')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Importance Weights', fontweight='bold')
axes[0].legend()

# Sorted weights
sorted_weights = torch.sort(weights)[0]
axes[1].plot(sorted_weights.numpy())
axes[1].axhline(y=1.0, color='red', linestyle='--', label='No shift (w=1)')
axes[1].set_xlabel('Sample Index (sorted)')
axes[1].set_ylabel('Importance Weight')
axes[1].set_title('Sorted Importance Weights', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nInterpretation: Samples with w > 1 are underrepresented in source but common in target.")
print("               Samples with w < 1 are overrepresented in source relative to target.")

In [ ]:
# Example: Using importance weights in training
print("Example: Weighted Loss Computation")
print("=" * 50)

# Simulate per-sample losses
dummy_losses = torch.rand(100)
dummy_weights = weights[:100]

# Standard loss (unweighted)
standard_loss = dummy_losses.mean()

# Weighted loss (adapts to target distribution)
weighted_loss = iw.weighted_loss(dummy_losses, dummy_weights)

print(f"Standard loss:  {standard_loss:.4f}")
print(f"Weighted loss:  {weighted_loss:.4f}")
print("\nThe weighted loss emphasizes samples that are more")
print("representative of the target distribution.")

## Part 10: Production Monitoring Simulation

## Part 9.5: Model Performance Degradation Under Shift

An important aspect of shift detection is understanding how model performance degrades. Let's measure accuracy at different shift severities.

In [ ]:
# Measure accuracy at different rotation levels
rotations = [0, 10, 20, 30, 45, 60]
accuracies = []
shift_scores = []

mmd_detector = detectors["MMD"]

print("Measuring accuracy vs shift severity...")
for rotation in rotations:
    if rotation == 0:
        transform_test = transform_normal
    else:
        transform_test = transforms.Compose([
            transforms.RandomRotation(degrees=rotation),
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,))
        ])
    
    test_dataset = datasets.FashionMNIST('./data', train=False, transform=transform_test)
    test_subset = Subset(test_dataset, range(1000))
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)
    
    # Compute accuracy
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs.to(device))
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels.to(device)).sum().item()
    
    accuracy = 100. * correct / total
    accuracies.append(accuracy)
    
    # Compute shift score
    features, _ = extract_model_features(model, test_loader, device)
    feature_loader = DataLoader(TensorDataset(features, torch.zeros(len(features))), batch_size=256)
    score = mmd_detector.score(feature_loader)
    shift_scores.append(score)
    
    print(f"  Rotation {rotation:2d}°: Accuracy = {accuracy:.1f}%, MMD = {score:.6f}")

print(f"\nAccuracy drop from 0° to 60°: {accuracies[0] - accuracies[-1]:.1f}%")

In [ ]:
# Visualize accuracy vs shift severity (dual y-axis)
fig, ax1 = plt.subplots(figsize=(10, 5))

# Accuracy on left axis
color1 = 'steelblue'
ax1.set_xlabel('Image Rotation (degrees)', fontsize=11)
ax1.set_ylabel('Accuracy (%)', color=color1, fontsize=11)
line1 = ax1.plot(rotations, accuracies, marker='o', linewidth=2, markersize=8, color=color1, label='Accuracy')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(0, 100)

# Shift score on right axis
ax2 = ax1.twinx()
color2 = 'crimson'
ax2.set_ylabel('MMD Shift Score', color=color2, fontsize=11)
line2 = ax2.plot(rotations, shift_scores, marker='s', linewidth=2, markersize=8, color=color2, label='MMD Score')
ax2.tick_params(axis='y', labelcolor=color2)

# Combined legend
lines = line1 + line2
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='center right')

ax1.set_title('Model Performance Degradation vs Shift Severity', fontweight='bold')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key insight: As MMD shift score increases, model accuracy decreases.")
print("This correlation validates using shift detection as an early warning for performance degradation.")

In [ ]:
# Simulate gradual shift over time (increasing rotation)
print("Simulating gradual covariate shift over time...")

rotations = [0, 10, 20, 30, 45, 60]
mmd_detector = detectors["MMD"]

# Compute baseline on rotation=0 (same distribution, different samples)
# This accounts for natural sampling variance
transform_baseline = transform_normal
baseline_dataset = datasets.FashionMNIST('./data', train=False, transform=transform_baseline)
baseline_subset = Subset(baseline_dataset, range(500))
baseline_loader = DataLoader(baseline_subset, batch_size=256, shuffle=False)
baseline_features, _ = extract_model_features(model, baseline_loader, device)
baseline_feature_loader = DataLoader(TensorDataset(baseline_features, torch.zeros(len(baseline_features))), batch_size=256)
baseline_score = mmd_detector.score(baseline_feature_loader)

print(f"Baseline (rotation 0°): {baseline_score:.6f}")
print(f"Warning threshold (3x):  {baseline_score * 3:.6f}")
print(f"Critical threshold (5x): {baseline_score * 5:.6f}")
print()

shifts_over_time = []

for rotation in rotations:
    if rotation == 0:
        transform_shift = transform_normal
    else:
        transform_shift = transforms.Compose([
            transforms.RandomRotation(degrees=rotation),
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,))
        ])
    
    shifted_dataset = datasets.FashionMNIST('./data', train=False, transform=transform_shift)
    shifted_subset = Subset(shifted_dataset, range(500, 1000))  # Different samples than baseline
    shifted_loader = DataLoader(shifted_subset, batch_size=256, shuffle=False)
    
    # Extract model features for the shifted data
    shifted_features, _ = extract_model_features(model, shifted_loader, device)
    shifted_feature_loader = DataLoader(TensorDataset(shifted_features, torch.zeros(len(shifted_features))), batch_size=256)
    
    shift_score = mmd_detector.score(shifted_feature_loader)
    shifts_over_time.append(shift_score)
    
    ratio = shift_score / baseline_score
    if ratio > 5:
        status = "CRITICAL"
    elif ratio > 3:
        status = "WARNING"
    else:
        status = "OK"
    print(f"  Rotation {rotation:2d}°: MMD = {shift_score:.6f} ({ratio:.1f}x) [{status}]")

In [ ]:
# Plot monitoring dashboard using library functions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Use the library's plot_shift_severity function
fig1 = plot_shift_severity(
    rotations, shifts_over_time,
    severity_label="Image Rotation (degrees)",
    score_label="MMD Shift Score",
    warning_threshold=baseline_score * 3,
    critical_threshold=baseline_score * 5,
    show=False
)
# Copy to combined figure
ax1_src = fig1.axes[0]
for line in ax1_src.get_lines():
    if line.get_linestyle() == '--':
        axes[0].axhline(y=line.get_ydata()[0], color=line.get_color(), linestyle='--', linewidth=2, label=line.get_label())
    else:
        axes[0].plot(line.get_xdata(), line.get_ydata(), marker='o', linewidth=2, markersize=8, color=line.get_color())
axes[0].fill_between(rotations, 0, baseline_score * 3, alpha=0.1, color='green')
axes[0].fill_between(rotations, baseline_score * 3, baseline_score * 5, alpha=0.1, color='orange')
axes[0].fill_between(rotations, baseline_score * 5, max(shifts_over_time) * 1.1, alpha=0.1, color='red')
axes[0].set_xlabel('Image Rotation (degrees)', fontsize=11)
axes[0].set_ylabel('MMD Shift Score', fontsize=11)
axes[0].set_title('Production Monitoring: Gradual Shift Detection', fontweight='bold')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)
plt.close(fig1)

# Detector comparison
ax = axes[1]
methods = list(results.keys())
x = np.arange(len(methods))
width = 0.35

normal_scores = [results[m]["normal"] for m in methods]
shifted_scores = [results[m]["shifted"] for m in methods]

ax.bar(x - width/2, normal_scores, width, label='No Shift', alpha=0.8, color='green')
ax.bar(x + width/2, shifted_scores, width, label='With Shift', alpha=0.8, color='red')
ax.set_xlabel('Detector')
ax.set_ylabel('Shift Score')
ax.set_title('Detector Comparison', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 11: Save and Load Detectors

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    # Save MMD detector
    mmd_path = f"{tmpdir}/mmd_detector.pt"
    detectors["MMD"].save(mmd_path)
    print(f"Saved MMD detector ({os.path.getsize(mmd_path)} bytes)")
    
    # Load and verify
    loaded_mmd = MMDShiftDetector()
    loaded_mmd.load_state_dict(torch.load(mmd_path, weights_only=True))
    
    original_score = detectors["MMD"].score(test_normal_feature_loader)
    loaded_score = loaded_mmd.score(test_normal_feature_loader)
    print(f"Original score: {original_score:.6f}")
    print(f"Loaded score:   {loaded_score:.6f}")
    print(f"Match: {abs(original_score - loaded_score) < 1e-6}")
    
    # Save label shift detector
    label_path = f"{tmpdir}/label_detector.pt"
    label_detector.save(label_path)
    print(f"\nSaved label shift detector ({os.path.getsize(label_path)} bytes)")
    
    # Save importance weighting
    iw_path = f"{tmpdir}/importance_weighting.pt"
    iw.save(iw_path)
    print(f"Saved importance weighting ({os.path.getsize(iw_path)} bytes)")

## Part 12: Production Deployment

In [ ]:
def monitor_shift(detector, new_data_loader, baseline_score, warning_threshold=1.5, critical_threshold=2.0):
    """
    Monitor for distribution shift in production.
    
    Args:
        detector: Fitted shift detector
        new_data_loader: New production data (feature loader)
        baseline_score: Score on clean reference data
        warning_threshold: Ratio for warning alert
        critical_threshold: Ratio for critical alert
    
    Returns:
        dict with shift info and recommended action
    """
    shift_score = detector.score(new_data_loader)
    shift_ratio = shift_score / (baseline_score + 1e-10)
    
    if shift_ratio > critical_threshold:
        alert = "CRITICAL"
        action = "Immediate retraining recommended"
    elif shift_ratio > warning_threshold:
        alert = "WARNING"
        action = "Monitor closely, plan retraining"
    else:
        alert = "OK"
        action = "No action needed"
    
    return {
        'shift_score': shift_score,
        'baseline_score': baseline_score,
        'shift_ratio': shift_ratio,
        'alert': alert,
        'action': action,
    }

# Example using model features
result = monitor_shift(detectors["MMD"], test_rotated_feature_loader, results["MMD"]["normal"])

print("Production Monitoring Example:")
print("=" * 60)
print(f"Baseline Score: {result['baseline_score']:.6f}")
print(f"Current Score:  {result['shift_score']:.6f}")
print(f"Shift Ratio:    {result['shift_ratio']:.2f}x")
print(f"Alert Level:    {result['alert']}")
print(f"Action:         {result['action']}")
print("=" * 60)

## Part 13: Best Practices

### When to Use Each Detector

| Scenario | Recommended Detector |
|----------|---------------------|
| Default choice | `MMDShiftDetector` |
| High-dimensional features | `ClassifierShiftDetector` (BBSD) |
| Per-feature analysis | `KSShiftDetector` |
| No kernel tuning | `EnergyShiftDetector` |
| Label distribution shift | `LabelShiftDetector` |
| Adapt to shift | `ImportanceWeightingShift` |

### Important: Use Model Features

**Always use model embeddings** for shift detection, not raw inputs:
- Extract features from penultimate layer of your trained model
- Lower dimensionality = more efficient and accurate
- Semantic features capture meaningful differences

### Threshold Selection

1. **Warning threshold**: 1.5x baseline (start monitoring closely)
2. **Critical threshold**: 2.0x baseline (trigger retraining)
3. **Adjust based on**: acceptable performance degradation, retraining cost, safety requirements

### Response Strategy

| Shift Level | Ratio | Action |
|-------------|-------|--------|
| Minor | 1.0-1.5x | Monitor, collect data |
| Moderate | 1.5-2.0x | Plan retraining, use importance weighting |
| Major | >2.0x | Immediate model update, investigate root cause |

### Production Checklist

- [ ] Deploy shift detector with model
- [ ] **Extract model features** for detection (not raw inputs)
- [ ] Set alert thresholds based on validation
- [ ] Log shift scores over time
- [ ] Automate retraining when shift detected
- [ ] Use multiple detectors (MMD + Classifier)
- [ ] Monitor both covariate and label shift
- [ ] Save detector state for consistent monitoring

## Summary

**What we learned:**
1. Distribution shift degrades model performance silently
2. **Covariate shift detectors**: MMD, Energy, KS, Classifier (BBSD)
3. **Label shift detection**: Estimate target distribution from predictions
4. **Importance weighting**: Adapt to covariate shift by reweighting samples
5. **Metrics**: Energy distance, Wasserstein, Sliced Wasserstein, TVD, PSI
6. **Visualizations**: Feature histograms, embedding space, confidence distributions, KS statistics, shift severity
7. **Performance correlation**: Shift scores correlate with accuracy degradation
8. **Production monitoring**: Continuous monitoring with automated alerts

**Key Takeaway:** Continuous monitoring for distribution shift is critical for maintaining model performance in production. Use multiple detection methods and respond appropriately based on shift severity.

**Next Steps:**
- Combine with calibration monitoring (notebook 01)
- Use with OOD detection (notebook 02)
- Implement automated retraining pipelines